In [ ]:
from conllu import parse, parse_incr
import pandas as pd
import os

In [2]:
DATASET_VERSION = 2.16
DATASET_PATH = f'D:/Files/Datasets/UD_Russian-SynTagRus-master_{DATASET_VERSION}'
CORPUS_TEXTS_FILEPATH = f'D:/Files/Datasets/UD_Russian-SynTagRus-master_{DATASET_VERSION}/corpus_{DATASET_VERSION}.txt'

In [3]:
train_df = pd.DataFrame()
test_df = pd.DataFrame()
dev_df = pd.DataFrame()

In [4]:
if DATASET_VERSION == 2.16:
    train_list = ['ru_syntagrus-ud-train-a.conllu', 'ru_syntagrus-ud-train-b.conllu', 'ru_syntagrus-ud-train-c.conllu']
else:
    train_list = ['ru_syntagrus-ud-train.conllu']

In [5]:
for dataset_name in train_list:
    with open(os.path.join(DATASET_PATH, dataset_name), 'r', encoding='utf-8') as data_file:
        parsed_list = list(parse_incr(data_file))

    source_text = []
    source_words = []
    lemmas = []
    upos = []
    xpos = []
    feats = []
    head = []
    deprel = []
    misc = []

    for seq in parsed_list:
        source_text.append(seq.metadata['text'])
        source_words.append([str(token) for token in seq])
        lemmas.append([grammem['lemma'] for grammem in seq])
        upos.append([grammem['upos'] for grammem in seq])
        xpos.append([grammem['xpos'] for grammem in seq])
        feats.append([grammem['feats'] for grammem in seq])
        head.append([grammem['head'] for grammem in seq])
        deprel.append([grammem['deprel'] for grammem in seq])
        misc.append([grammem['misc'] for grammem in seq])

    temp_df = pd.DataFrame({
    'source_text': source_text,
    'source_words': source_words,
    'lemmas': lemmas,
    'upos': upos,
    'xpos': xpos,
    'feats': feats,
    'head': head,
    'deprel': deprel,
    'misc': misc
    }, dtype='object')

    print(len(temp_df))

    train_df = pd.concat((train_df, temp_df))

for datapath in ['ru_syntagrus-ud-test.conllu', 'ru_syntagrus-ud-dev.conllu']:
    data_file = open(os.path.join(DATASET_PATH, datapath), 'r', encoding='utf-8')
    parsed_list = list(parse_incr(data_file))

    source_text = []
    source_words = []
    lemmas = []
    upos = []
    xpos = []
    feats = []
    head = []
    deprel = []
    misc = []

    for seq in parsed_list:
        source_text.append(seq.metadata['text'])
        source_words.append([str(token) for token in seq])
        lemmas.append([grammem['lemma'] for grammem in seq])
        upos.append([grammem['upos'] for grammem in seq])
        xpos.append([grammem['xpos'] for grammem in seq])
        feats.append([grammem['feats'] for grammem in seq])
        head.append([grammem['head'] for grammem in seq])
        deprel.append([grammem['deprel'] for grammem in seq])
        misc.append([grammem['misc'] for grammem in seq])

    temp_df = pd.DataFrame({
    'source_text': source_text,
    'source_words': source_words,
    'lemmas': lemmas,
    'upos': upos,
    'xpos': xpos,
    'feats': feats,
    'head': head,
    'deprel': deprel,
    'misc': misc
    }, dtype='object')

    print(len(temp_df))

    if datapath == 'ru_syntagrus-ud-test.conllu':
        test_df = temp_df
    else:
        dev_df = temp_df

24516
24299
20816
8800
8906


In [6]:
def unfold_nested_elements(df, feat_column:str='feats'):
    # Выполняем предварительный проход для поиска всех грамматических атрибутов
    nested_features = set()
    
    # Используем iterrows() для безопасного доступа к строкам
    for idx, row in df.iterrows():
        df_row = row[feat_column]
        # Проверяем, что значение является списком
        if isinstance(df_row, list):
            for item in df_row:
                if isinstance(item, dict):
                    for feature in item.keys():
                        nested_features.add(feature)
    
    # Заполняем None для всех грамматических атрибутов
    for feature in nested_features:
        if feature not in df.columns:
            # Создаем список значений для каждого слова в предложении
            df[feature] = df[feat_column].apply(
                lambda x: ['None'] * len(x) if isinstance(x, list) else []
            )
    
    # Заполняем реальные значения для грамматических атрибутов
    for idx, row in df.iterrows():
        df_row = row[feat_column]
        if isinstance(df_row, list):
            for dict_idx, item in enumerate(df_row):
                if isinstance(item, dict):
                    for feature, value in item.items():
                        # Обновляем значение для конкретной позиции
                        current_list = df.at[idx, feature]
                        if isinstance(current_list, list) and dict_idx < len(current_list):
                            current_list[dict_idx] = value
                            df.at[idx, feature] = current_list
    
    return df

In [7]:
train_df = train_df.reset_index()
train_df = unfold_nested_elements(train_df)

test_df = test_df.reset_index()
test_df = unfold_nested_elements(test_df)

dev_df = dev_df.reset_index()
dev_df = unfold_nested_elements(dev_df)

In [8]:
print(len(train_df))
print(len(test_df))
print(len(dev_df))

69631
8800
8906


In [9]:
# Создание текстового файла всех исходных текстов обучающей выборки
with open(CORPUS_TEXTS_FILEPATH, "w", encoding="utf-8") as file:
    for raw_text in train_df['source_text']:
        file.write(raw_text + '\n')

    for raw_text in test_df['source_text']:
        file.write(raw_text + '\n')

    for raw_text in dev_df['source_text']:
        file.write(raw_text + '\n')

In [10]:
train_df.to_parquet(os.path.join(DATASET_PATH, 'ru_syntagrus-ud-train.parquet'), engine="fastparquet", index=False)
test_df.to_parquet(os.path.join(DATASET_PATH, 'ru_syntagrus-ud-test.parquet'), engine="fastparquet", index=False)
dev_df.to_parquet(os.path.join(DATASET_PATH, 'ru_syntagrus-ud-dev.parquet'), engine="fastparquet", index=False)